# 🚀 NeoTenet Persistent Workspace & Hermes Agent Deployment

> محیط Ubuntu روی Google Colab با Google Drive Backup، Web Terminal و Cloudflare Tunnel

این Notebook کد پروژه‌ی NeoTenet را اجرا می‌کند و پس از اجرا لینک دسترسی Web Terminal را نمایش می‌دهد.

## ▶️ اجرا

کل سلول کد زیر را اجرا کنید. در اولین اجرا، Google Colab برای اتصال به Google Drive درخواست دسترسی خواهد کرد.

In [ ]:
import os
import re
import time
import threading
import subprocess
from pathlib import Path

class NeoTenetEnvironment:
    def __init__(self):
        self.port = 7681
        self.log_file = Path("/tmp/cloudflared.log")
        self.bashrc_path = Path("/root/.bashrc")
        self.drive_dir = Path("/content/drive/MyDrive/Hermes_Project")
        self.hermes_path = Path("/root/hermes")

    def mount_drive_and_restore(self):
        print("📂 [Phase 1] Mounting Google Drive & Restoring Backup...")
        try:
            from google.colab import drive
            drive.mount('/content/drive', force_remount=False)
            self.drive_dir.mkdir(parents=True, exist_ok=True)
            
            backup_file = self.drive_dir / "hermes_backup.tar.gz"
            if backup_file.exists():
                print("📦 Found previous Hermes backup! Restoring files...")
                subprocess.run(f"tar -xzvf {backup_file} -C / > /dev/null 2>&1", shell=True)
                print("✅ Hermes Workspace successfully restored!")
            else:
                print("ℹ️ No previous backup found. A new workspace will be initialized.")
        except Exception as e:
            print(f"⚠️ Drive Mount Warning: {e}")

    def configure_shell_prompt(self):
        # تغییر مسیر به root، تنظیم پرامپت روت و بنر خوانا و شیک
        prompt_config = r"""
cd /root
export PATH="/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:$PATH"
export PS1="\[\e[1;32m\]NeoTenet\[\e[0m\]@\[\e[1;34m\]colab\[\e[0m\]:\[\e[1;33m\]~\[\e[0m\]# "
clear
echo -e "\e[1;35m  =======================================================\e[0m"
echo -e "\e[1;36m   🚀  NEOTENET TERMINAL WORKSPACE | ROOT ACCESS ACTIVE  \e[0m"
echo -e "\e[1;35m  =======================================================\e[0m"
echo -e "\e[1;32m   📢 Telegram Channel : @NeoTenet\e[0m"
echo -e "\e[1;33m   ⚡ Auto-Backup      : Active (Syncing to Google Drive)\e[0m"
echo -e "\e[1;35m  =======================================================\e[0m"
echo ""
"""
        self.bashrc_path.write_text(prompt_config)

    def prepare_dependencies(self):
        print("⚡ [Phase 2] Optimizing repositories & fetching packages...")
        
        # پاکسازی مخازن اضافی برای رفع هشدار apt
        subprocess.run("sed -i '/stat.illinois.edu/d' /etc/apt/sources.list /etc/apt/sources.list.d/* 2>/dev/null", shell=True)
        
        # نصب ابزارها
        packages = ["neofetch", "htop", "curl", "wget", "git", "python3-pip", "tmate"]
        subprocess.run(f"apt-get update -qq && apt-get install -y {' '.join(packages)} -qq", shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

        # دانلود مستقیم باینری‌ها
        binaries = {
            "/usr/local/bin/ttyd": "https://github.com/tsl0922/ttyd/releases/download/1.7.7/ttyd.x86_64",
            "/tmp/cloudflared": "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
        }
        for dest, url in binaries.items():
            subprocess.run(f"wget -q -nc {url} -O {dest} && chmod +x {dest}", shell=True)

    def start_background_autobackup(self):
        """ سرویس بکاپ‌گیری خودکار در پس‌زمینه (هر ۵ دقیقه) """
        def backup_loop():
            while True:
                time.sleep(300) # هر ۵ دقیقه یک بار
                if self.drive_dir.exists():
                    target = "/root/hermes" if self.hermes_path.exists() else "/root"
                    backup_file = self.drive_dir / "hermes_backup.tar.gz"
                    cmd = f"tar -czvf {backup_file} {target} > /dev/null 2>&1"
                    subprocess.run(cmd, shell=True)

        thread = threading.Thread(target=backup_loop, daemon=True)
        thread.start()
        print("🔄 [Background Service] Auto-backup daemon started successfully.")

    def terminate_old_processes(self):
        for proc in ["ttyd", "cloudflared", "tmate"]:
            subprocess.run(["pkill", "-9", proc], stderr=subprocess.DEVNULL)
        time.sleep(1)

    def initialize_services(self):
        print("🌐 [Phase 3] Deploying web console & cloudflare tunnel...")
        
        # اجرای ttyd با تم مشکی
        ttyd_args = [
            "ttyd", "-W", "-p", str(self.port),
            "-t", "fontSize=16",
            "-t", "theme={'background': '#000000', 'foreground': '#ffffff'}",
            "bash", "--rcfile", str(self.bashrc_path)
        ]
        subprocess.Popen(ttyd_args)
        time.sleep(1)

        # ساخت تونل
        subprocess.Popen(f"/tmp/cloudflared tunnel --url http://localhost:{self.port} > {self.log_file} 2>&1", shell=True)

    def fetch_endpoint_url(self):
        endpoint = None
        for _ in range(30):
            time.sleep(1)
            if self.log_file.exists():
                logs = self.log_file.read_text()
                found = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', logs)
                if found:
                    endpoint = found[0]
                    break
        return endpoint

    def start(self):
        self.mount_drive_and_restore()
        self.prepare_dependencies()
        self.configure_shell_prompt()
        self.start_background_autobackup()
        self.terminate_old_processes()
        self.initialize_services()
        
        url = self.fetch_endpoint_url()
        
        print("\n" + "─"*55)
        print("  💎 NEOTENET ENGINE IS ACTIVE & AUTO-SYNCED")
        print("─"*55)
        print(f"  🔗 ACCESS LINK : {url if url else 'Failed to generate tunnel'}")
        print("  📂 BACKUP DIR  : /content/drive/MyDrive/Hermes_Project")
        print("─"*55)

        try:
            while True:
                time.sleep(60)
        except KeyboardInterrupt:
            print("\nShutting down NeoTenet Environment...")

if __name__ == "__main__":
    app = NeoTenetEnvironment()
    app.start()

## 📢 NeoTenet

کانال رسمی: [@NeoTenet](https://t.me/NeoTenet)